# FIQA + saliency 결과 compact

기존 batch의 **단일 분할·20분할 보고서 전체**를 로컬에서 검증·집계합니다. 원본 노트북·CSV·manifest·실험 결과는 수정하지 않습니다.
첫 코드 셀 설정 후 **Kernel Restart → Run All** 하세요. 새 compact만 별도 디렉터리에 저장합니다. GPU 추론·보정 재적합·bootstrap 재실행은 없습니다.

조건별 정보량과 비교 맥락을 우선합니다. 32KB는 참고값이며 초과했다고 삭제하거나 강제 분할하지 않습니다. 기본 조건별 metrics 35행, paired 90행은 한 파일로 유지하고, 페이지당 최대 150행으로 설정합니다. 노트북에는 경로·coverage·짧은 안내만 출력합니다.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                    if (p / "research").is_dir() and (p / ".git").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SOURCE_REPORT_ROOT = PROJECT_ROOT / "results/calibration/matrix/reports"
# Explicit current report inventory; never select an automatic latest report.
REPORT_IDS = (
    "matrix-report-2e2ab02ddb1d039ef1084383",  # single seed 8972
    "matrix-report-d690567c6f4f7e7ad44ef19e",  # all 20 seeds
)
REQUIRE_ALL_CURRENT_REPORTS = True
OUTPUT_ROOT = PROJECT_ROOT / "results/calibration/matrix_compact"
PREFERRED_FILE_BYTES = 32_000  # advisory: preserve information above this size
ROWS_PER_PAGE = 150
PREVIEW_MAX_BYTES = 6_000  # only the notebook preview is bounded
PREVIEW_FILE = "START_HERE.md"


## 1. 입력과 범위 확인

현재 보고서 목록과 명시한 REPORT_IDS가 다르면 중단합니다. 추가 실행 보고서를 포함하려면 목록을 갱신하세요. 완료 상태·파일 SHA·36조건·방법·목표·seed·분모·paired 수치·split 요약을 검증한 뒤 집계합니다. 두 보고서를 합쳐 21개 seed처럼 계산하지 않습니다.

In [2]:
from scripts.compact_calibration_results import build_compact, preview_file, verify_compact

available = {p.parent.name for p in SOURCE_REPORT_ROOT.glob("*/manifest.json")}
if REQUIRE_ALL_CURRENT_REPORTS and set(REPORT_IDS) != available:
    raise ValueError(f"보고서 목록 갱신 필요. 미포함={sorted(available - set(REPORT_IDS))}; 누락={sorted(set(REPORT_IDS) - available)}")
print(f"검증·집계 대상: {len(REPORT_IDS)}개 보고서. 결과는 로컬에서 처리합니다.")


검증·집계 대상: 2개 보고서. 결과는 로컬에서 처리합니다.


## 2. 별도 compact 생성

START_HERE → 데이터셋별 목표 충족 overview → 조건별 metrics/paired 순으로 읽습니다. 모든 원본 행을 집계에 사용하며, 상세 seed·모델 계수·query 기록의 무손실 사본은 기존 원본/증거 ZIP에 남아 있습니다.
원본 입력 해시와 compact 구현 해시로 새 UID를 만들고, 같은 UID가 있으면 무결성 검증 후 재사용합니다.

In [3]:
compact_dir, compact_manifest = build_compact(
    [SOURCE_REPORT_ROOT / name for name in REPORT_IDS], OUTPUT_ROOT,
    preferred_file_bytes=PREFERRED_FILE_BYTES, max_rows=ROWS_PER_PAGE,
)
verify_compact(compact_dir)
print(f"저장 경로: {compact_dir}")
for source in compact_manifest["spec"]["sources"]:
    print(f"seed {len(source['seeds'])}개: metric {source['metric_rows']:,}행 / paired {source['paired_rows']:,}행 전체 집계")
print(f"파일 {compact_manifest['file_count']}개, 참고 크기 초과 {compact_manifest['above_preferred_files']}개 (내용 보존)")


저장 경로: C:\ronbun\results\calibration\matrix_compact\compact-5a51fbb472bfb777625df8aa
seed 1개: metric 1,260행 / paired 3,240행 전체 집계
seed 20개: metric 25,200행 / paired 64,800행 전체 집계
파일 161개, 참고 크기 초과 0개 (내용 보존)


## 3. 채팅용 진입점만 미리보기

ChatGPT에는 START_HERE.md를 먼저 읽도록 요청하고, 필요한 데이터셋/조건의 파일 한 개씩 읽으세요. 전체 폴더를 한꺼번에 출력하면 compact여도 대화 한도를 넘을 수 있습니다. 아래 셀은 큰 파일을 자동으로 잘라 출력하지 않고, 경로 안내만 제공합니다.

FPIR·TPIR 범위는 기술통계이고 seed는 독립 반복이 아닙니다. CI 끝점 범위를 통합 CI로 해석하지 마세요. 두 방법이 목표를 충족해도 실제 FPIR가 동일한 것은 아닙니다.

In [4]:
print(preview_file(compact_dir, PREVIEW_FILE, max_bytes=PREVIEW_MAX_BYTES))

# FIQA + saliency compact

로컬 Python으로 원본 전체를 검증·집계했습니다. 아래 보고서를 하나씩 읽으세요.
먼저 [지표 설명](METRICS.md)을 확인하세요. FPIR 목표 실패를 숨기거나 TPIR 개선만으로 방법을 선정하지 않았습니다.

- [1개 seed 보고서](matrix-report-2e2ab02ddb1d039ef1084383/README.md): metric 1,260행, paired 3,240행 → 조건별 compact.
- [20개 seed 보고서](matrix-report-d690567c6f4f7e7ad44ef19e/README.md): metric 25,200행, paired 64,800행 → 조건별 compact.

